In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.market_loader import MarketLoader

from src.curves.curve_snapshot import CurveSnapshot
from src.curves.bootstrap.bootstrap_engine import BootstrapCurveEngine
from src.curves.projection_curve import ProjectionCurve
from src.curves.zero_curve import ZeroCurve
from src.curves.simulator.hull_white_2factor import HullWhite2FactorSimulator
from src.curves.simulator.hull_white_2factor_pricer import HullWhite2FactorPricer
from src.curves.calibrator.pca_factor_extractor import PCAFactorExtractor
from src.curves.calibrator.ou_factor_calibrator import OUFactorCalibrator

from src.pricing.hull_white_2factor_swap_pricer import HullWhite2FactorSwapPricer

from src.instruments.instrument_builder import InstrumentBuilder

from src.trades.interest_rate_swap import InterestRateSwap

from src.risk.exposure.stochastic_exposure_engine_2factor import MonteCarloExposureEngine2Factor
from src.risk.exposure.collateralized_exposure_engine import CollateralizedMonteCarloExposureEngine

from src.risk.collateral.csa import CSAAgreement
from src.risk.collateral.netting_set import NettingSet

In [2]:
# downloading market curves
market_loader = MarketLoader()
market_curves = market_loader.market_loader_pipeline()

# downloading swap curves
swap_loader = MarketLoader()
swap_curves = swap_loader.swap_loader_pipeline()

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..
estr curve dataset already downloaded..
usd_ois curve dataset already downloaded..
eur_ois curve dataset already downloaded..


In [3]:
# yield curve history
curve_history = market_curves['treasury']
curve_history = curve_history / 100

In [4]:
### create curve snapshots
# SOFR snapshot
sofr_df = market_curves['sofr']

latest_date = sofr_df.index[-1]
latest_sofr_curve = sofr_df.iloc[-1]

sofr_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'sofr',
    as_of_date = latest_date,
    curve_row = latest_sofr_curve
)

# Futures snapshot
future_df = market_curves['futures']
latest_futures_curve = future_df.iloc[-1]

futures_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'futures',
    as_of_date = latest_date,
    curve_row = latest_futures_curve
)

# OIS snapshot
ois_df = swap_curves['usd_ois']

latest_swap_date = ois_df.index[-1]
latest_swap_curve = ois_df.iloc[-1]

ois_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'usd_ois',
    as_of_date = latest_swap_date,
    curve_row = latest_swap_curve
)

In [5]:
### create instruments from curve snapshot
# deposits
deposit_instruments = InstrumentBuilder.build_deposit_instruments(snapshot = sofr_snapshot)

# futures
future_instruments = InstrumentBuilder.build_future_instruments(snapshot = futures_snapshot)

# ois
ois_instruments = InstrumentBuilder.build_ois_instruments(snapshot = ois_snapshot)

### discount, projection and zero curve builder
# bootstrapping engine for generating the discount curve
all_instruments = deposit_instruments + future_instruments + ois_instruments

engine = BootstrapCurveEngine()

discount_curve = engine.bootstrap(
    snapshot = sofr_snapshot,
    instruments = all_instruments
)

# projection curve
projection_curve = ProjectionCurve(discount_curve = discount_curve)

# zero curve
zero_curve = ZeroCurve(discount_curve = discount_curve)

In [6]:
# sample IRS trade objects
IR_swap1 = InterestRateSwap(
    notional = 1_000_000,
    maturity = 3.0,
    fixed_rate = 3.40,
    pay_fixed = True
)

IR_swap2 = InterestRateSwap(
    notional = 2_500_000,
    maturity = 5.0,
    fixed_rate = 3.75,
    pay_fixed = False
)

IR_swap3 = InterestRateSwap(
    notional = 1_500_000,
    maturity = 2.0,
    fixed_rate = 3.25,
    pay_fixed = True
)

In [7]:
### PCA factor extractor
# initialize PCA factor extractor
pca = PCAFactorExtractor(n_factors = 2)

# fitting PCAs to historical yield changes and projecting onto factors
factor_history = pca.fit_transform(yield_curve_history = curve_history)

In [8]:
# Ornstein-Uhlenbeck factor calibration
ou_calibrator = OUFactorCalibrator(dt = 1/252)

# calibrate HW 2-factor model parameters
hw2f_params = ou_calibrator.calibrate_hw2f(factor_history = factor_history)
hw2f_params

{'a': 0.0994245389493929,
 'b': 0.06493166163063027,
 'sigma1': 0.018724138829696178,
 'sigma2': 0.008874071471247505,
 'rho': -0.0006194015774373465}

In [9]:
### HW 2-factor simulator using calibrated HW 2-factor model parameters
# short rate
r0 = zero_curve.get_zero_rate(maturity = 0.25)

# simulator engine
hw_simulator = HullWhite2FactorSimulator(
    r0 = r0,
    a = hw2f_params['a'],
    b = hw2f_params['b'],
    sigma1 = hw2f_params['sigma1'],
    sigma2 = hw2f_params['sigma2'],
    rho = hw2f_params['rho'],
    random_seed = 2
)

# hw pricer
hw_pricer = HullWhite2FactorPricer(
    zero_curve = zero_curve,
    a = hw2f_params['a'],
    b = hw2f_params['b'],
    sigma1 = hw2f_params['sigma1'],
    sigma2 = hw2f_params['sigma2'],
    rho = hw2f_params['rho']
)

# hw 2-factor swap pricer
hw_swap_pricer = HullWhite2FactorSwapPricer(hw_pricer = hw_pricer)

# exposure engine
hw2f_trade_engine = MonteCarloExposureEngine2Factor(
    pricer = hw_swap_pricer,
    simulator = hw_simulator
)

In [10]:
### Collateralization framework
# CSA agreement
H = 150_000
MTA = 25_000
IA = 100_000
MPOR = 10
trade_list = [
    IR_swap1,
    IR_swap2,
    IR_swap3
]

csa = CSAAgreement(
    threshold = H,
    minimum_transfer_amount = MTA,
    independent_amount = IA,
    margin_period_of_risk = MPOR
)

print(csa.summary())

# netting set
netting_set = NettingSet(
    trades = trade_list,
    csa_agreement = csa,
    netting_id = 'USD_IRS_portfolio_1'
)

display(netting_set.summary())

# collateralized engine
collateral_engine = CollateralizedMonteCarloExposureEngine(
    trade_exposure_engine = hw2f_trade_engine,
    netting_engine = netting_set
)


{'Threshold': 150000, 'MTA': 25000, 'IndependentAmount': 100000, 'MPOR_Days': 10}


,NettingID,Trades,CSAEnabled
0,USD_IRS_portfolio_1,3,True


In [11]:
# collateralized expected exposure
collateralized_ee = collateral_engine.expected_exposure(
    n_paths = 250,
    steps_per_year = 4
)
collateralized_ee

,Times,CollateralizedEE
0,0.00,0.000000
1,0.25,9583.350548
2,0.50,11522.039296
3,0.75,13863.903511
4,1.00,14313.502115
5,1.25,13186.192041
6,1.50,13710.265473
7,1.75,13257.276705
8,2.00,12831.986776
9,2.25,13347.536542


In [12]:
# collateralized potential exposure
collateralized_pfe = collateral_engine.potential_future_exposure(
    percentile = 95.0,
    n_paths = 250,
    steps_per_year = 4
)
collateralized_pfe

,Times,PFE_95%
0,0.00,0.000000
1,0.25,50000.000000
2,0.50,50000.000000
3,0.75,50000.000000
4,1.00,50000.000000
5,1.25,50000.000000
6,1.50,50000.000000
7,1.75,50000.000000
8,2.00,50000.000000
9,2.25,50000.000000


In [13]:
# netting benefit
netting_benefit = collateral_engine.netting_benefit(
    n_paths = 250,
    steps_per_year = 4
)
netting_benefit

,Times,NettingBenefit
0,0.00,0.000000
1,0.25,15991.513806
2,0.50,20194.264345
3,0.75,20461.229893
4,1.00,20860.367830
5,1.25,20220.477307
6,1.50,17629.181577
7,1.75,14228.344165
8,2.00,8875.643459
9,2.25,8020.163038
